# Replication Notebook — ICU Example, Data Construction (Step 1 of 3)

This notebook prepares the ICU dataset used in Section 3 (and Appendix C.1) of the paper

*“The choice of reference group can reverse conclusions in the Oaxaca–Blinder decomposition.”*

This notebook is **Step 1** of the replication pipeline. It builds the clean analysis dataset (`ICU_clean.csv`) from the raw PhysioNet files. No modeling or decomposition is performed here.

## Replication pipeline

The `Real-data example/` folder is **self‐contained**: it includes the data‐construction notebook, the two model notebooks, all input / output CSVs, and an environment specification. Running the three notebooks below — in order, top to bottom — reproduces all main‐text and appendix results for the ICU example.

| Step | Notebook | Reproduces |
|------|----------|-----------|
| 1 | `construct_ICU_data.ipynb` | Builds `ICU_clean.csv` from raw PhysioNet files. |
| 2 | `icu_139_final_models.ipynb` | Table 1 (HR quartile 2 coefficient table, 4 models) and the three flip‐summary tables (main text + appendix) for **linear, logistic, neural net, XGBoost**. |
| 3 | `icu_139_tabpfn_local.ipynb` | TabPFN row of Table 1 and the three flip‐summary tables (run on a local TabPFN install, CPU or GPU). |

## Environment

Python ≥ 3.10. Install dependencies with:

```bash
pip install -r requirements.txt
```

Equivalent one‐line install:

```bash
pip install numpy==1.26.* pandas==2.2.* scipy==1.13.* scikit-learn==1.5.* \
            statsmodels==0.14.* matplotlib==3.8.* xgboost==2.1.* joblib==1.4.* tabpfn==2.0.*
```

A jupyter kernel is required to execute the notebooks (`pip install jupyter` or use VS Code / JupyterLab). All notebooks run end‐to‐end without manual intervention; expected run times on a modern laptop (CPU only):

- Step 1: < 2 minutes (file I/O over ~4,000 patient time‐series files).
- Step 2: ~25 minutes for the four models with **B = 1000** bootstrap replications, parallelised.
- Step 3 (CPU): several hours; **~10 minutes on a single GPU** with `TABPFN_DEVICE="cuda"`.

## Data access (PhysioNet Challenge 2012, set‐a)

The ICU data are public but not redistributed in this repository. To download:

1. Create a free account on PhysioNet and accept the data‐use agreement for the *Predicting Mortality of ICU Patients: The PhysioNet/Computing in Cardiology Challenge 2012* dataset (training set‐a).
2. Download the outcomes file `Outcomes-a.txt` and the patient time‐series archive `set-a.tar.gz`.
3. Extract `set-a.tar.gz` so that each patient is a `.txt` file under `set-a/set-a/`.

After downloading, the `Real-data example/` folder should look like:

```
Real-data example/
├── requirements.txt
├── construct_ICU_data.ipynb        # Step 1 — produces ICU_clean.csv
├── icu_139_final_models.ipynb      # Step 2 — 4 models
├── icu_139_tabpfn_local.ipynb      # Step 3 — TabPFN
├── ICU_clean.csv                   # produced by Step 1 (output, not shipped)
└── archive/                        # PhysioNet raw data (downloaded by the user)
    ├── Outcomes-a.txt
    └── set-a/
        └── set-a/
            ├── 132539.txt
            ├── 132540.txt
            └── ...
```

To reproduce `ICU_clean.csv`, place the PhysioNet files as shown above and run this notebook top to bottom. The CSV is then loaded by Steps 2 and 3.

## Construction of the analysis dataset

- Each ICU stay is collapsed into a **single row**.
- For each physiological variable, we take the **first observed value** in the time‐series.
- We merge with the outcomes file using `RecordID`.
- Outcome: in‐hospital mortality. Group indicator: gender (female vs. male; `Gender == -1` is excluded).
- We restrict to observations with non‐missing values in the covariates `[Age, ICUType, HR, NIMAP, Temp, Urine]`.

The resulting `ICU_clean.csv` contains one row per patient, the outcome, the group indicator, and the six covariates.


In [1]:
import os
import numpy as np
import pandas as pd

## Load and clean ICU data


In [ ]:
# Path to the Outcomes file
outcomes_path = "archive/Outcomes-a.txt"

# Outcomes file is comma-separated
outcomes = pd.read_csv(outcomes_path, sep=",", header=0)

print("Outcomes shape:", outcomes.shape)
outcomes.head()


In [ ]:
# Build one wide row per ICU stay
data_dir = "archive/set-a/set-a"

if not os.path.isdir(data_dir):
    raise FileNotFoundError(
        f"Directory '{data_dir}' not found. "
        "Make sure the PhysioNet data are downloaded and placed "
        "in the correct folder structure."
    )

records = []

for fname in os.listdir(data_dir):
    if not fname.endswith(".txt"):
        continue

    record_id = os.path.splitext(fname)[0]  # e.g., "132539"
    path = os.path.join(data_dir, fname)

    # Each file is a CSV with columns: Time, Parameter, Value
    ts = pd.read_csv(path)

    # Ensure 'Value' is numeric where possible
    ts["Value"] = pd.to_numeric(ts["Value"], errors="coerce")

    # Take the first observed value for each Parameter
    first_vals = ts.groupby("Parameter")["Value"].first()

    entry = first_vals.to_dict()
    entry["RecordID"] = int(record_id)

    records.append(entry)

df_raw = pd.DataFrame(records)

print("Raw ICU table shape:", df_raw.shape)
df_raw.head()


In [ ]:

# Merge with outcomes, restrict to analysis sample,
# define outcome/covariates, and drop missing values

# Ensure RecordID types match
outcomes["RecordID"] = pd.to_numeric(outcomes["RecordID"], errors="coerce")

df = df_raw.merge(outcomes, on="RecordID", how="inner")
print("Merged dataset shape:", df.shape)

# Restrict to patients with known gender and define female indicator
df = df[df["Gender"] != -1].copy()
df["female"] = (df["Gender"] == 1).astype(int)

# Outcome and covariates 
y_col = "In-hospital_death"
x_cols = ["Age", "ICUType", "HR", "NIMAP", "Temp", "Urine"]

# Drop missing values
ICU_clean = df.dropna(subset=x_cols + [y_col, "female"]).copy()
print("Final analysis sample size:", ICU_clean.shape)


In [5]:
ICU_clean.to_csv("ICU_clean.csv", index=False)